In [1]:
import os
import io
import pickle
from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import sys

In [3]:
def authenticate():
    import pickle
    from google.auth.transport.requests import Request
    from google_auth_oauthlib.flow import InstalledAppFlow

    SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
    creds = None

    if os.path.exists('token.pickle'):
        with open('token.pickle', 'rb') as token:
            creds = pickle.load(token)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_console()  # ✅ Jupyter-friendly auth
        with open('token.pickle', 'wb') as token:
            pickle.dump(creds, token)

    return creds

In [5]:
def list_folder_contents(service, folder_id, parent_path='PhdFiles', drive_id=None, file_counter=None):
    import sys
    from googleapiclient.http import MediaIoBaseDownload

    os.makedirs(parent_path, exist_ok=True)

    if file_counter is None:
        file_counter = {'total': 0, 'current': 0}

    query = f"'{folder_id}' in parents and trashed = false"
    list_kwargs = {
        'q': query,
        'fields': "files(id, name, mimeType)",
        'supportsAllDrives': True,
        'includeItemsFromAllDrives': True
    }

    if drive_id:
        list_kwargs['corpora'] = 'drive'
        list_kwargs['driveId'] = drive_id
    else:
        list_kwargs['corpora'] = 'user'

    # Only calculate total file count once
    if file_counter['total'] == 0:
        def count_files(folder_id):
            count = 0
            sub_kwargs = list_kwargs.copy()
            sub_kwargs['q'] = f"'{folder_id}' in parents and trashed = false"
            sub_results = service.files().list(**sub_kwargs).execute()
            for item in sub_results.get('files', []):
                if item['mimeType'] == 'application/vnd.google-apps.folder':
                    count += count_files(item['id'])
                else:
                    count += 1
            return count

        file_counter['total'] = count_files(folder_id)
        print(f"\n🔢 Total files to download: {file_counter['total']}\n")

    results = service.files().list(**list_kwargs).execute()
    items = results.get('files', [])

    if not items:
        print("No files found in this folder.")
    else:
        for item in items:
            name = item['name']
            file_id = item['id']
            mime_type = item['mimeType']

            if mime_type == 'application/vnd.google-apps.folder':
                new_path = os.path.join(parent_path, name)
                list_folder_contents(service, file_id, new_path, drive_id, file_counter)
            else:
                file_counter['current'] += 1
                print(f"\n📥 [{file_counter['current']} / {file_counter['total']}] Downloading: {name}")
                file_path = os.path.join(parent_path, name)

                request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
                with io.FileIO(file_path, 'wb') as fh:
                    downloader = MediaIoBaseDownload(fh, request)
                    done = False
                    while not done:
                        status, done = downloader.next_chunk()
                        progress = int(status.progress() * 100)
                        sys.stdout.write(f"\r    {progress}% complete")
                        sys.stdout.flush()
                print()  # Newline after progress

In [7]:
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
FOLDER_ID = '18BMH-PhTZNiN8rbXyqZvz5J9qRGrRqbA'  # just the ID
DRIVE_ID = '0AHMeY0Fo1d79Uk9PVA'
CUSTOM_DOWNLOAD_PATH = r'C:\Users\Ry\Documents\PhD Thesis Writing\Data'

In [9]:
creds = authenticate()
service = build('drive', 'v3', credentials=creds)
list_folder_contents(service, FOLDER_ID,parent_path=CUSTOM_DOWNLOAD_PATH ,drive_id=DRIVE_ID)


🔢 Total files to download: 194


📥 [1 / 194] Downloading: measurement_23_07_25_ramp1ts.npy
    100% complete

📥 [2 / 194] Downloading: measurement_23_07_25_ramp1xs.npy
    100% complete

📥 [3 / 194] Downloading: measurement_23_07_25_ramp1ns.npy
    100% complete

📥 [4 / 194] Downloading: measurement_23_07_25_constant1xs.npy
    100% complete

📥 [5 / 194] Downloading: measurement_23_07_25_constant1ts.npy
    100% complete

📥 [6 / 194] Downloading: measurement_23_07_25_constant1ns.npy
    100% complete

📥 [7 / 194] Downloading: measurement_23_07_25_varying1ts.npy
    100% complete

📥 [8 / 194] Downloading: measurement_23_07_25_varying1ns.npy
    100% complete

📥 [9 / 194] Downloading: measurement_23_07_25_varying1xs.npy
    100% complete

📥 [10 / 194] Downloading: measurement_23_07_25_test11xs.npy
    100% complete

📥 [11 / 194] Downloading: measurement_23_07_25_test11ts.npy
    100% complete

📥 [12 / 194] Downloading: measurement_23_07_25_test11ns.npy
    100% complete

📥 [13 / 194] Do

KeyboardInterrupt: 